# Fractal Analysis: Hausdorff Dimension, IFS, and Multifractal Spectra

This notebook investigates fractal geometry through three lenses:

1. **Box-counting dimension**: numerical estimate of Hausdorff dimension for classical fractals
2. **Iterated Function Systems (IFS)**: generating fractals from affine contractions
3. **Multifractal spectra**: generalized dimensions D_q describing the scaling of measures

## Classical Fractals Studied
- Cantor set: D = log(2)/log(3) ~ 0.6309
- Koch snowflake curve: D = log(4)/log(3) ~ 1.2619
- Sierpinski triangle: D = log(3)/log(2) ~ 1.5850
- Barnsley fern: D ~ 1.8928 (IFS attractor)

All computations are self-contained using only numpy.

In [ ]:
import sys
from pathlib import Path


sys.path.insert(0, str(Path.cwd().parent.parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec


try:
    import pandas as pd

    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("pandas not available")

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 10)
np.random.seed(42)
print("Setup complete.")

## 1. Generating Classical Fractals

We generate point-set approximations of each fractal.

In [ ]:
# -------------------------------------------------------------------
# Cantor set: keep middle-thirds out, n_iterations deep
# -------------------------------------------------------------------
def cantor_set(n_iter):
    """Return array of (x, y=0) points approximating the Cantor set.
    Start with [0,1], remove middle thirds iteratively.
    At level n, we have 2^n intervals of length 3^{-n}.
    We sample 8 points per interval.
    """
    intervals = [(0.0, 1.0)]
    for _ in range(n_iter):
        new_intervals = []
        for a, b in intervals:
            third = (b - a) / 3.0
            new_intervals.append((a, a + third))
            new_intervals.append((b - third, b))
        intervals = new_intervals

    pts = []
    for a, b in intervals:
        for t in np.linspace(a, b, 8):
            pts.append([t, 0.0])
    return np.array(pts)


# -------------------------------------------------------------------
# Koch curve: line from (0,0) to (1,0), n_iter steps
# -------------------------------------------------------------------
def koch_curve(n_iter):
    """Return ordered array of (x, y) vertices of the Koch curve."""
    pts = np.array([[0.0, 0.0], [1.0, 0.0]])
    for _ in range(n_iter):
        new_pts = [pts[0]]
        for i in range(len(pts) - 1):
            a = pts[i]
            b = pts[i + 1]
            p1 = a + (b - a) / 3.0
            a + 2.0 * (b - a) / 3.0
            # Apex of the equilateral triangle
            mid = (a + b) / 2.0
            perp = np.array([-(b[1] - a[1]), b[0] - a[0]]) / 3.0
            mid + perp * np.sqrt(3) / 2.0
            # Correction: standard Koch peak
            d = b - a
            p1 + np.array([-d[1], d[0]]) / (np.sqrt(3))
            # Recompute properly
            pa = a + d / 3.0
            pb = a + 2.0 * d / 3.0
            # peak: rotate d/3 by 60 degrees CCW from pa
            angle = np.pi / 3.0
            rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
            peak = pa + rot @ (d / 3.0)
            new_pts.extend([pa, peak, pb])
        new_pts.append(pts[-1])
        pts = np.array(new_pts)
    return pts


# -------------------------------------------------------------------
# Sierpinski triangle: chaos game
# -------------------------------------------------------------------
def sierpinski_triangle(n_pts):
    """Generate Sierpinski triangle via the chaos game."""
    vertices = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, np.sqrt(3) / 2]])
    pts = np.zeros((n_pts, 2))
    p = np.array([0.5, 0.3])
    for i in range(n_pts):
        v = vertices[np.random.randint(3)]
        p = (p + v) / 2.0
        pts[i] = p
    return pts


cantor_pts = cantor_set(6)
koch_pts = koch_curve(5)
sierp_pts = sierpinski_triangle(50000)

print(f"Cantor set points: {len(cantor_pts)}")
print(f"Koch curve vertices: {len(koch_pts)}")
print(f"Sierpinski triangle points: {len(sierp_pts)}")

## 2. Box-Counting Dimension

The box-counting dimension is estimated by:
  D = lim_{eps->0} log N(eps) / log(1/eps)

where N(eps) is the number of boxes of side eps needed to cover the set.
We compute the slope of log N vs log(1/eps) via linear regression.

In [ ]:
def box_counting_dimension(pts, n_scales=20, min_boxes=4, max_boxes=512):
    """Estimate box-counting dimension of a point set.

    Parameters
    ----------
    pts : ndarray shape (N, d)
        Point coordinates.
    n_scales : int
        Number of box scales to use.

    Returns
    -------
    dimension : float
        Estimated box-counting dimension.
    log_eps_inv : ndarray
        log(1/epsilon) values used.
    log_N : ndarray
        log(N(epsilon)) values.
    """
    pts = np.atleast_2d(pts)
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    ranges = maxs - mins
    max_range = ranges.max()

    # Scales from max_range / max_boxes to max_range / min_boxes
    eps_values = np.logspace(
        np.log10(max_range / max_boxes), np.log10(max_range / min_boxes), n_scales
    )

    log_eps_inv = []
    log_N = []

    for eps in eps_values:
        # Digitize each point into a box
        indices = np.floor((pts - mins) / eps).astype(int)
        n_boxes = len(set(map(tuple, indices)))
        log_eps_inv.append(np.log(1.0 / eps))
        log_N.append(np.log(n_boxes))

    log_eps_inv = np.array(log_eps_inv)
    log_N = np.array(log_N)

    # Linear regression
    coeffs = np.polyfit(log_eps_inv, log_N, 1)
    dimension = coeffs[0]
    return dimension, log_eps_inv, log_N


# Cantor set (1D points, but 2D array with y=0)
cantor_1d = cantor_pts[:, :1]  # use only x
dim_cantor, lx_c, lN_c = box_counting_dimension(cantor_1d, n_scales=15)

# Koch curve
dim_koch, lx_k, lN_k = box_counting_dimension(koch_pts, n_scales=15)

# Sierpinski triangle
dim_sierp, lx_s, lN_s = box_counting_dimension(sierp_pts, n_scales=15)

theory_cantor = np.log(2) / np.log(3)
theory_koch = np.log(4) / np.log(3)
theory_sierp = np.log(3) / np.log(2)

print("Box-counting dimension estimates:")
print(f"  Cantor set:   computed={dim_cantor:.4f}  theoretical={theory_cantor:.4f}")
print(f"  Koch curve:   computed={dim_koch:.4f}  theoretical={theory_koch:.4f}")
print(f"  Sierpinski:   computed={dim_sierp:.4f}  theoretical={theory_sierp:.4f}")

## 3. Barnsley Fern via IFS

The Barnsley fern is the attractor of 4 affine transformations.
Each iteration picks one transformation at random (with given probabilities)
and maps the current point.

In [ ]:
def barnsley_fern(n_pts=100000):
    """Generate Barnsley fern via IFS with 4 affine maps.

    The four transforms (a,b,c,d,e,f) with probability p:
      f1: stem         p=0.01
      f2: leaflet CCW  p=0.85
      f3: small left   p=0.07
      f4: small right  p=0.07
    Each transform: x' = a*x + b*y + e, y' = c*x + d*y + f
    """
    # Each tuple stores six affine coefficients followed by probability.
    transforms = [
        (0.00, 0.00, 0.00, 0.16, 0.00, 0.00, 0.01),  # stem
        (0.85, 0.04, -0.04, 0.85, 0.00, 1.60, 0.85),  # main leaflet
        (0.20, -0.26, 0.23, 0.22, 0.00, 1.60, 0.07),  # left leaflet
        (-0.15, 0.28, 0.26, 0.24, 0.00, 0.44, 0.07),  # right leaflet
    ]

    probs = np.array([t[6] for t in transforms])
    probs /= probs.sum()
    cum_probs = np.cumsum(probs)

    pts = np.zeros((n_pts, 2))
    x, y = 0.0, 0.0

    rand_vals = np.random.random(n_pts)

    for i in range(n_pts):
        r = rand_vals[i]
        if r < cum_probs[0]:
            a, b, c, d, e, f, _ = transforms[0]
        elif r < cum_probs[1]:
            a, b, c, d, e, f, _ = transforms[1]
        elif r < cum_probs[2]:
            a, b, c, d, e, f, _ = transforms[2]
        else:
            a, b, c, d, e, f, _ = transforms[3]
        x_new = a * x + b * y + e
        y_new = c * x + d * y + f
        x, y = x_new, y_new
        pts[i] = [x, y]

    return pts


fern_pts = barnsley_fern(n_pts=80000)
print(f"Barnsley fern: {len(fern_pts)} points generated")
print(f"  x range: [{fern_pts[:, 0].min():.3f}, {fern_pts[:, 0].max():.3f}]")
print(f"  y range: [{fern_pts[:, 1].min():.3f}, {fern_pts[:, 1].max():.3f}]")

dim_fern, lx_f, lN_f = box_counting_dimension(fern_pts, n_scales=18)
print(f"  Box-counting dimension: {dim_fern:.4f}  (theoretical ~1.8928)")

## 4. Multifractal Spectrum

The generalized dimensions D_q characterize a multifractal measure:
  D_q = (1/(q-1)) * lim_{eps->0} log sum_i p_i^q / log eps

For q=0: D_0 = box-counting dimension
For q=1: D_1 = information dimension (via L'Hopital)
For q=2: D_2 = correlation dimension

We compute D_q for a simple multiplicative multifractal measure
(binomial cascade on [0,1] with p0=0.4, p1=0.6).

In [ ]:
def multifractal_Dq_binomial(p0, p1, q_values):
    """Compute generalized dimensions D_q for a binomial cascade measure.

    At level n, the measure on interval [k/2^n, (k+1)/2^n] is
    p0^{n-s} * p1^s where s = number of 1-bits in k.

    For the binomial cascade:
      sum_i mu_i^q = (p0^q + p1^q)^n
      sum_i mu_i^q ~ eps^{(q-1)*D_q} with eps = 2^{-n}
    So D_q = log(p0^q + p1^q) / ((q-1) * log(1/2)) for q != 1.
    """
    D_q = []
    for q in q_values:
        if abs(q - 1) < 1e-10:
            # D_1 = information dimension = -(p0 log p0 + p1 log p1) / log(2)
            D1 = -(p0 * np.log(p0) + p1 * np.log(p1)) / np.log(2)
            D_q.append(D1)
        else:
            val = np.log(p0**q + p1**q) / ((q - 1) * np.log(0.5))
            D_q.append(val)
    return np.array(D_q)


p0, p1 = 0.4, 0.6
q_range = np.linspace(-5, 5, 51)
Dq = multifractal_Dq_binomial(p0, p1, q_range)

print(f"Binomial multifractal (p0={p0}, p1={p1}):")
print(
    f"  D_0 = {Dq[np.argmin(np.abs(q_range))]::.4f}  (box-counting, expected 1.0 since support = [0,1])"
)
print(f"  D_1 = {Dq[np.argmin(np.abs(q_range - 1.0))]::.4f}  (information dimension)")
print(f"  D_2 = {Dq[np.argmin(np.abs(q_range - 2.0))]::.4f}  (correlation dimension)")
print(f"  D_q is monotone decreasing: {bool(np.all(np.diff(Dq) <= 1e-10))}")

## 5. Verification Table

In [ ]:
data = [
    {
        "Set": "Cantor set",
        "Theoretical_dim": round(theory_cantor, 4),
        "Computed_dim": round(dim_cantor, 4),
        "Error_pct": round(abs(dim_cantor - theory_cantor) / theory_cantor * 100, 2),
    },
    {
        "Set": "Koch curve",
        "Theoretical_dim": round(theory_koch, 4),
        "Computed_dim": round(dim_koch, 4),
        "Error_pct": round(abs(dim_koch - theory_koch) / theory_koch * 100, 2),
    },
    {
        "Set": "Sierpinski triangle",
        "Theoretical_dim": round(theory_sierp, 4),
        "Computed_dim": round(dim_sierp, 4),
        "Error_pct": round(abs(dim_sierp - theory_sierp) / theory_sierp * 100, 2),
    },
    {
        "Set": "Barnsley fern",
        "Theoretical_dim": 1.8928,
        "Computed_dim": round(dim_fern, 4),
        "Error_pct": round(abs(dim_fern - 1.8928) / 1.8928 * 100, 2),
    },
]

if HAS_PANDAS:
    import pandas as pd

    df = pd.DataFrame(data)
    print("Fractal Dimension Verification Table")
    print(df.to_string(index=False))
else:
    print(f"{'Set':22} {'Theory':12} {'Computed':12} {'Error%':8}")
    for r in data:
        print(
            f"{r['Set']:22} {r['Theoretical_dim']:12.4f} {r['Computed_dim']:12.4f} {r['Error_pct']:8.2f}"
        )

## 6. Visualization

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Cantor set (show first 5 iterations)
ax1 = fig.add_subplot(gs[0, 0])
cantor_iters = []
for n in range(6):
    pts_n = cantor_set(n)
    cantor_iters.append(pts_n)
    ax1.plot(
        pts_n[:, 0], np.full(len(pts_n), n), "|", markersize=4, color=plt.cm.Blues(0.3 + 0.12 * n)
    )
ax1.set_xlabel("x")
ax1.set_ylabel("Iteration")
ax1.set_title(f"Cantor Set (D={theory_cantor:.4f})", fontsize=11)
ax1.set_yticks(range(6))
ax1.grid(True, alpha=0.2, axis="x")

# Panel 2: Koch curve
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(koch_pts[:, 0], koch_pts[:, 1], "b-", linewidth=0.5)
ax2.set_title(f"Koch Curve (D={theory_koch:.4f})", fontsize=11)
ax2.set_aspect("equal")
ax2.axis("off")

# Panel 3: Sierpinski triangle
ax3 = fig.add_subplot(gs[0, 2])
ax3.scatter(sierp_pts[::5, 0], sierp_pts[::5, 1], s=0.1, c="navy", alpha=0.5)
ax3.set_title(f"Sierpinski Triangle (D={theory_sierp:.4f})", fontsize=11)
ax3.set_aspect("equal")
ax3.axis("off")

# Panel 4: Barnsley fern
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(fern_pts[::4, 0], fern_pts[::4, 1], s=0.2, c="forestgreen", alpha=0.4)
ax4.set_title("Barnsley Fern (D~1.8928)", fontsize=11)
ax4.set_aspect("equal")
ax4.axis("off")

# Panel 5: Box-counting log-log plots
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(lx_c, lN_c, "o-", label=f"Cantor D={dim_cantor:.3f}", markersize=4)
ax5.plot(lx_k, lN_k, "s-", label=f"Koch D={dim_koch:.3f}", markersize=4)
ax5.plot(lx_s, lN_s, "^-", label=f"Sierp D={dim_sierp:.3f}", markersize=4)
ax5.set_xlabel("log(1/eps)")
ax5.set_ylabel("log N(eps)")
ax5.set_title("Box-Counting Log-Log Plots", fontsize=11)
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3)

# Panel 6: D_q spectrum
ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(q_range, Dq, "purple", linewidth=2)
ax6.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="D=1 (uniform)")
ax6.axvline(0, color="k", linewidth=0.5)
ax6.axvline(1, color="orange", linewidth=0.8, linestyle=":", label="q=1 (info dim)")
ax6.axvline(2, color="green", linewidth=0.8, linestyle=":", label="q=2 (corr dim)")
ax6.set_xlabel("q")
ax6.set_ylabel("D_q")
ax6.set_title(f"Multifractal D_q: binomial (p0={p0}, p1={p1})", fontsize=11)
ax6.legend(fontsize=8)
ax6.grid(True, alpha=0.3)

plt.savefig("fractal_analysis.png", dpi=120, bbox_inches="tight")
plt.show()

## Conclusion

This notebook demonstrated:

1. **Box-counting dimension** numerically estimated for classical fractals, recovering theoretical values within a few percent using 15-18 scale steps.
2. **Iterated Function Systems**: the Barnsley fern generated from 4 affine contractions, with IFS attractor dimension ~1.89.
3. **Multifractal spectrum D_q**: the binomial cascade shows D_q monotone decreasing from D_{-inf} to D_{+inf}, with D_0 = 1 (topological), D_1 = entropy dimension, D_2 = correlation dimension.

The multifractal formalism connects to the turbulence analysis in notebook 06, where the energy cascade in 2D turbulence produces a multifractal measure on wavenumber space.